Lien vers le notebook dans databricks  : https://dbc-b0984f6a-a4b8.cloud.databricks.com/editor/notebooks/1072870576869292?o=560273488971544

In [0]:
filepath='s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json'

### Tidying

In [0]:
data_raw=spark.read.format('json').load(filepath)
data_raw.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField
from typing import List, Dict, Generator, Union, Callable

# This is actually written like a scala function, we'll walk you through it
def walkSchema(schema: Union[StructType, StructField]) -> Generator[str, None, None]:
    """Explores a PySpark schema:
    
    schema: StructType | StructField
    
    Yield
    -----
    A generator of strings, the name of each field in the schema
    """
    
    # we define a function _walk that produces a string generator from
    # a dictionnary "schema_dct", and a string "prefix"
    def _walk(schema_dct: Dict['str', Union['str', list, dict]],
              prefix: str = "") -> Generator[str, None, None]:
        assert isinstance(prefix, str), "prefix should be a string" # check if prefix is a string
        
        # this function returns "name" if there's no prefix and "prefix.name" if prefix exists
        fullName: Callable[str, str] = lambda name: ( 
            name if not prefix else f"{prefix}.{name}")
        
        # we get the next name one level lower from the dictionnary
        name = schema_dct.get('name', '')
        
        # if the type is struct then we search for the fields key
        # if fields is there we apply the function again and dig one level deeper in
        # the schema and set a prefix
        if schema_dct['type'] == 'struct':
            assert 'fields' in schema_dct, (
                "It's a StructType, we should have some fields")
            for field in schema_dct['fields']:
                yield from _walk(field, prefix=prefix)
        # if we have a dict type and we can't find fields then we
        # dig one level deeper and apply the _walk function again
        elif isinstance(schema_dct['type'], dict):
            assert 'fields' not in schema_dct, (
                "We're missing some keys here")
            yield from _walk(schema_dct['type'], prefix=fullName(name))
        # If we finally reached the end and found a name we yield the full name
        elif name:
            # name=name.replace(' ','_').replace('.','_').replace('-','_').replace('1','un').replace('2','deux').replace('3','trois').replace('4','quatre').replace('5','cing').replace('6','siw').replace('7','sept').replace('8','huit')
            yield fullName(name)
    
    yield from _walk(schema.jsonValue())

# yield as opposed to return, returns a result but does not stop the function from running, it keeps
# running even after returning one result.

In [0]:
for x in walkSchema(data_raw.schema):
    print(x)

In [0]:
from pyspark.sql import functions as F

data_tidy = data_raw
for name in walkSchema(data_tidy.schema):
    if "data.tags" in name:
        continue
    else:
        data_tidy = data_tidy.withColumn(name.split(".")[-1], F.col(name))

# Convertir en Pandas DataFrame
data_tidy = data_tidy.drop("data")
data_tidy.limit(10).toPandas()

In [0]:
display(data_tidy)

### preprocessing

In [0]:
data_tidy.filter(F.col("id") != F.col("appid")).count()

In [0]:
data_tidy = data_tidy.drop("appid")

In [0]:
data_tidy = (
    data_tidy.withColumn("price", F.col("price").cast("float") / 100)
    .withColumn("initialprice", F.col("price").cast("float") / 100)
    .withColumn("discount", F.col("price").cast("int"))
)

In [0]:
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.max_columns", None)
data_tidy.select(
    *(F.sum(F.col(c).isNull().cast("int")).alias(c) for c in data_tidy.columns)
).toPandas().set_index(pd.Index(["missing"]))

In [0]:
data_tidy.columns

### Which publisher has released the most games on Steam?

In [0]:
data_publisher = data_tidy.groupBy("publisher").count().orderBy(F.desc(F.col("count")))
display(
    data_publisher.filter(F.col("count") > 90).filter(
        (F.col("publisher").isNotNull())
        & (F.col("publisher") != "")
        & (F.col("publisher") != " ")
    )
)

### What are the best rated games?

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, desc, rank, expr

window_spec = Window.orderBy(col("positive"))

data_ratings = (
    data_tidy.withColumn(
        "ratings_ratio", expr("try_divide(positive, positive + negative)")
    )
    .withColumn(
        "ratings_nb_positiveratings", rank().over(window_spec) / data_tidy.count()
    )
    .withColumn(
        "ratings_score", col("ratings_ratio") * col("ratings_nb_positiveratings")
    )
    .select(
        col("name"),
        col("negative"),
        col("positive"),
        col("ratings_ratio"),
        col("ratings_nb_positiveratings"),
        col("ratings_score"),
        col("genre"),
        col("type"),
    )
    .orderBy(desc(col("ratings_score")))
)

In [0]:
### Cherche la valeur du rating score pour n'avoir que les 100 première lignes

score30 = (
    data_ratings.orderBy(F.desc("ratings_score"))
    .limit(50)
    .toPandas()
    .ratings_score[29]
    .round(4)
)
score30

In [0]:
display(data_ratings.filter(F.col("ratings_score") > score30))

### Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
display(data_tidy)

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

data_year = (
    data_tidy.withColumn(
        "release_ts", F.expr("try_to_timestamp(release_date, 'yyyy/MM/dd')")
    )
    .withColumn("year", F.year("release_ts"))
    .withColumn(
        "covid",
        F.when((F.col("year") == 2020) | (F.col("year") == 2021), "Covid").otherwise(
            "No Covid"
        ),
    )
    .select("year", "id", "covid")
    .orderBy("year")
    .filter(F.col("year").isNotNull())
)

In [0]:
display(data_year)

### How are the prizes distributed? Are there many games with a discount?

In [0]:
data_price = data_tidy.select(F.col("initialprice"), F.col("price"), F.col("discount"))

In [0]:
display(data_price.filter(F.col("price") < 150))

In [0]:
display(
    data_price.select("discount").withColumn(
        "discount_has",
        F.when(F.col("discount") > 0, "hasDiscount").otherwise("hasNotDiscount"),
    )
)

### What are the most represented languages?

In [0]:
data_language = data_tidy.select("id", "languages")

In [0]:
from pyspark.sql import functions as F

# Supposons que data_language soit votre DataFrame d'origine
words_to_remove = [
    "(full",
    "(all",
    " ",
    "audio)",
    "Not",
    "(text",
    "full",
    "-",
    "only)",
    "audio",
    "support)",
    "[b]*[/b]",
    "Simplified",
    "",
    "supported",
    "with",
    "Traditional",
]

# Appliquer toutes les transformations dans une seule chaîne de `withColumn`
df_modified = (
    data_language
    # Étape 1 : Diviser la colonne 'languages' par des virgules
    .withColumn("language", F.explode(F.split(F.col("languages"), ",")))
    # Étape 2 : Supprimer les espaces et nettoyer les balises HTML
    .withColumn("language", F.explode(F.split(F.trim(F.col("language")), " ")))
    .withColumn("language", F.explode(F.split(F.col("language"), "\n")))
    # Étape 3 : Filtrer les lignes qui ne contiennent pas les mots de la liste
    .filter(~F.col("language").isin(words_to_remove))
    # Étape 4 : Nettoyer la colonne 'language' pour supprimer les balises et caractères spéciaux
    .withColumn(
        "language", F.regexp_replace(F.col("language"), r"\[.*?\]|\*|;|\(|\)", "")
    )
    # Supprimer les lignes vides ou nulles après nettoyage
    .filter(F.col("language").isNotNull() & (F.col("language") != ""))
    .drop("languages")
)

In [0]:
display(
    df_modified.groupBy("language").count().orderBy("count").filter(F.col("count") > 50)
)

### Are there many games prohibited for children under 16/18?

In [0]:
display(data_tidy.select("required_age").distinct())

In [0]:
data_age = data_tidy.select("required_age")

In [0]:
data_age_cleaned = (
    data_age.withColumn(
        "required_age", F.regexp_replace(F.col("required_age"), r"(\+)", "")
    )
    .withColumn("required_age", F.regexp_replace(F.col("required_age"), r"(MA )", ""))
    .withColumn("required_age", F.regexp_replace(F.col("required_age"), r"(180)", "18"))
    .withColumn("required_age", F.col("required_age").cast("float"))
)

In [0]:
data_age_cleaned = data_age_cleaned.select(F.col("required_age")).withColumn(
    "sup16", F.when(F.col("required_age") > 16, "sup 16").otherwise("inf 16")
)
display(data_age_cleaned)

### What are the most represented genres?

In [0]:
display(data_tidy.select("genre"))

In [0]:
df_genre_tidy = (
    data_tidy.select(F.col("name"), F.col("genre"))
    .withColumn("genre2", F.explode(F.split(F.col("genre"), ", ")))
    .filter(F.col("genre2") != "")
)

In [0]:
display(df_genre_tidy.groupBy("genre2").count().orderBy("count"))

### Are there any genres that have a better positive/negative review ratio?

In [0]:
df_join = data_ratings.join(df_genre_tidy, on="name").drop("genre")
display(
    df_join.groupBy("genre2")
    .agg(F.mean(F.col("ratings_score")).alias("mean_ratings_coef"))
    .orderBy(F.desc("mean_ratings_coef"))
)